## Lab 2 (Weeks 5–6): Conformed Date & Time Dimensions and Slowly Changing Dimensions

> Before starting, review these resources:
> - `lab2_intro.md` — concepts covered in lecture
> - `merge_intro.md` — how to write a `MERGE INTO` statement (required for Tasks 3 and 4)
> - `dim_date_intro.md` — how to use the Snowflake `GENERATOR` function to build a date dimension (required for Tasks 1 and 2)

### What you are building
This lab extends Lab 1 by:
- Adding `dim_date` and `dim_time` as conformed, reusable dimensions
- Refactoring the fact table to replace raw dates with surrogate date/time keys
- Upgrading `dim_account` to **SCD Type 2** (full history)
- Upgrading `dim_product` to **SCD Type 3** (current + prior family)

### Naming convention
All table names must end with your `_LAST_FI` suffix (e.g., `_clark_c`).
Save this notebook as `LAST_FI_LAB2` in the `MODELED` schema (e.g., `CLARK_C_LAB2`).

---
## Setup
Run this cell first before anything else.

In [ ]:
USE DATABASE SNOWBEARAIR_DB;
USE SCHEMA MODELED;
USE WAREHOUSE RAW_WH;

---
# Part 1: Design — Create the New and Updated Tables

**Build order:**
1. `dim_date` (no dependencies)
2. `dim_time` (no dependencies)
3. `dim_account` — rebuild with Type 2 columns
4. `dim_product` — rebuild with Type 3 column
5. `fact_opportunity_line_item` — rebuild with date/time foreign keys

---
### Task 1: Create dim_date

Create `dim_date_last_fi`

**Requirements:**
- `date_key` as INTEGER — formatted as `YYYYMMDD` (e.g., `20240315` for March 15 2024). **Do not use AUTOINCREMENT** — the date itself is the key.
- One row per calendar date, covering at least 2015–2025
- At least 20 descriptive attributes (see the list in `lab2.md`)
- **SCD Type 0** — rows are never updated once inserted

**Tip:** dim_date is populated by *generating* rows, not by querying Salesforce.

In [ ]:
-- Task 1: Create dim_date_last_fi

CREATE OR REPLACE TABLE dim_date_last_fi (
    date_key             INTEGER       PRIMARY KEY,  -- YYYYMMDD format
    calendar_date        DATE          NOT NULL,
    -- add your 20+ descriptive attributes below

);

---
### Task 2: Create dim_time

Create `dim_time_last_fi`

**Requirements:**
- `time_key` as INTEGER — formatted as `HHMM` (e.g., `1430` for 2:30 PM)
- One row per minute of the day (1,440 rows total)
- Required attributes: `hour_value`, `minute_value`, `second_value`, `am_pm_indicator`, `business_hours_flag`

In [ ]:
-- Task 2: Create dim_time_last_fi

CREATE OR REPLACE TABLE dim_time_last_fi (
    time_key             INTEGER       PRIMARY KEY,  -- HHMM format
    -- add remaining attributes

);

---
### Task 3: Upgrade dim_account to SCD Type 2

Rebuild `dim_account_last_fi` with full historical tracking.

**Add these three administrative columns to your Lab 1 structure:**
- `effective_start_date DATE` — first date this row's profile was valid
- `effective_end_date DATE` — last date valid; use `'9999-12-31'` for the current row (avoid NULLs)
- `current_flag BOOLEAN` — `TRUE` for the one active row per account

Use `CREATE OR REPLACE TABLE` — do not ALTER the Lab 1 table.

In [ ]:
-- Task 3: Rebuild dim_account_last_fi with Type 2 columns

CREATE OR REPLACE TABLE dim_account_last_fi (
    account_key          INTEGER        AUTOINCREMENT PRIMARY KEY,
    account_id           VARCHAR(18)    NOT NULL,
    -- your descriptive attributes from Lab 1

    -- Type 2 administrative columns (add these)
    effective_start_date DATE           NOT NULL,
    effective_end_date   DATE           NOT NULL DEFAULT '9999-12-31',
    current_flag         BOOLEAN        NOT NULL DEFAULT TRUE
);

---
### Task 4: Upgrade dim_product to SCD Type 3

Rebuild `dim_product_last_fi` to track current and prior product family.

**Add one column:**
- `prior_product_family_name VARCHAR(100)` — the family before the most recent reclassification

All other attributes continue using Type 1 behavior (overwrite on change).

In [ ]:
-- Task 4: Rebuild dim_product_last_fi with Type 3 prior column

CREATE OR REPLACE TABLE dim_product_last_fi (
    product_key               INTEGER       AUTOINCREMENT PRIMARY KEY,
    product_id                VARCHAR(18)   NOT NULL,
    -- your descriptive attributes from Lab 1

    -- Type 3 column (add this)
    prior_product_family_name VARCHAR(100)
);

---
### Task 5: Refactor the Fact Table

Rebuild `fact_opportunity_line_item_last_fi` with role-playing date keys.

**Requirements:**
- Remove any raw DATE/TIMESTAMP columns
- Add `created_date_key` and `close_date_key` — both reference `dim_date_last_fi` (role-playing)
- Add `created_time_key` — references `dim_time_last_fi`
- Keep all existing FKs and measures from Lab 1

**Role-playing reminder:** You are NOT creating two separate date tables. Both date keys point to the same `dim_date_last_fi` table — just with different column names.

In [ ]:
-- Task 5: Rebuild fact_opportunity_line_item_last_fi with date/time keys

CREATE OR REPLACE TABLE fact_opportunity_line_item_last_fi (
    -- your columns
);

---
# Part 2: Load Data

**Load order:** date → time → account → product → fact

`dim_date` and `dim_time` are generated from scratch — not queried from Salesforce.
`dim_account` and `dim_product` are reloaded from `SNOWBEARAIR_DB.PUBLIC` with new columns.
The fact table must be reloaded with joins to the date dimension to resolve date keys.

---
### Load dim_date

Generate rows using Snowflake's `GENERATOR` function — do **not** query from Salesforce.

See `dim_date_intro.md` for the `GENERATOR` + `SEQ4()` + `DATEADD` pattern and tips for calculating each attribute.

In [ ]:
-- Load dim_date_last_fi using the Snowflake GENERATOR function
-- See dim_date_intro.md for the full pattern and attribute expressions

INSERT INTO dim_date_last_fi (
    -- your columns
)
SELECT
    -- your expressions
FROM TABLE(GENERATOR(ROWCOUNT => /* number of days to generate */));

---
### Load dim_time

Generate all 1,440 rows (one per minute of the day) using `GENERATOR`.
Apply the same GENERATOR pattern from `dim_date_intro.md` — adapt the math to produce hour and minute values instead of calendar dates.

In [ ]:
-- Load dim_time_last_fi — 1,440 rows (one per minute)

INSERT INTO dim_time_last_fi (
    -- your columns
)
SELECT
    -- your expressions
FROM TABLE(GENERATOR(ROWCOUNT => 1440));

---
### Load dim_account (Type 2 — MERGE)

Use `MERGE INTO` to load `dim_account_last_fi`.

- `WHEN NOT MATCHED` — insert new accounts with `current_flag = TRUE`, `effective_end_date = '9999-12-31'`
- `WHEN MATCHED` — for changed accounts, use the two-step expire + insert pattern described in `merge_intro.md`

> Match on the natural key (`account_id` ↔ source `id`).

In [ ]:
-- Load dim_account_last_fi using MERGE INTO (Type 2 pattern)
-- See merge_intro.md for the expire + insert approach

MERGE INTO dim_account_last_fi AS target
USING SNOWBEARAIR_DB.PUBLIC.account AS source
    ON -- your match condition
WHEN MATCHED THEN UPDATE SET
    -- update descriptive attributes
WHEN NOT MATCHED THEN INSERT (
    -- your columns
) VALUES (
    -- source values; set effective_start_date and current_flag defaults here
);

---
### Load dim_product (Type 3 — MERGE)

Use `MERGE INTO` to load `dim_product_last_fi`.

- `WHEN NOT MATCHED` — insert new products with `prior_product_family_name = NULL`
- `WHEN MATCHED` — update attributes; when the family changes, shift current → prior before writing the new value

> Match on the natural key (`product_id` ↔ source `id`). See `merge_intro.md` for the Type 3 update pattern.

In [ ]:
-- Load dim_product_last_fi using MERGE INTO (Type 3 pattern)
-- See merge_intro.md for the Type 3 update pattern

MERGE INTO dim_product_last_fi AS target
USING SNOWBEARAIR_DB.PUBLIC.product AS source
    ON -- your match condition
WHEN MATCHED THEN UPDATE SET
    -- shift current family to prior, then write new current value
WHEN NOT MATCHED THEN INSERT (
    -- your columns
) VALUES (
    -- source values; set prior_product_family_name = NULL
);

---
### Reload fact_opportunity_line_item

Reload the fact table with the new date/time key columns.

**Date key resolution:** Convert the source date to YYYYMMDD integer format and join to `dim_date_last_fi` — same pattern shown in `dim_date_intro.md`.

**Role-playing joins:** Join `dim_date_last_fi` twice with different aliases — once for `created_date_key`, once for `close_date_key`.

**Type 2 join note:** Add `AND a.current_flag = TRUE` when joining `dim_account_last_fi` to get the current account profile.

In [ ]:
-- Reload fact_opportunity_line_item_last_fi with date/time surrogate keys

INSERT INTO fact_opportunity_line_item_last_fi (
    -- your columns
)
SELECT
    -- surrogate keys from dimension tables
    -- date keys resolved from dim_date
    -- measures from oli
FROM SNOWBEARAIR_DB.PUBLIC.opportunity_line_item oli
-- your JOINs to resolve surrogate keys including role-playing date joins
;

---
## Verify Your Work

Run these queries to confirm correctness before submitting.

In [ ]:
-- Row counts
SELECT 'dim_date'                      AS table_name, COUNT(*) AS row_count FROM dim_date_last_fi
UNION ALL SELECT 'dim_time',                          COUNT(*)              FROM dim_time_last_fi
UNION ALL SELECT 'dim_account',                       COUNT(*)              FROM dim_account_last_fi
UNION ALL SELECT 'dim_product',                       COUNT(*)              FROM dim_product_last_fi
UNION ALL SELECT 'fact_opportunity_line_item',        COUNT(*)              FROM fact_opportunity_line_item_last_fi;

-- Verify dim_date coverage
-- SELECT MIN(calendar_date), MAX(calendar_date), COUNT(*) FROM dim_date_last_fi;

-- Verify Type 2: exactly 1 current row per account
-- SELECT account_id, SUM(CASE WHEN current_flag THEN 1 ELSE 0 END) AS current_rows
-- FROM dim_account_last_fi GROUP BY account_id HAVING current_rows != 1;

-- Revenue by year and quarter using date dimension (role-playing)
-- SELECT cd.year, cd.quarter, SUM(f.total_price) AS revenue
-- FROM fact_opportunity_line_item_last_fi f
-- JOIN dim_date_last_fi cd ON cd.date_key = f.created_date_key
-- GROUP BY cd.year, cd.quarter ORDER BY cd.year, cd.quarter;

---
## Deliverables Checklist

- [ ] Notebook saved as `LAST_FI_LAB2` in the `MODELED` schema
- [ ] All table names include your `_LAST_FI` suffix
- [ ] `dim_date` has at least 20 attributes and covers 2015–2025
- [ ] `dim_time` has 1,440 rows (one per minute)
- [ ] `dim_account` has Type 2 columns with correct initial-load defaults
- [ ] `dim_product` has `prior_product_family_name` column (NULL on initial load)
- [ ] Fact table has `created_date_key` and `close_date_key` (no raw date columns)
- [ ] All load cells run without errors
- [ ] Row counts verified
- [ ] Star schema diagram uploaded **or** full DDL written (either/or)
- [ ] Written explanation in Canvas covering:
  - Why Type 2 for `dim_account` and not Type 1
  - Why Type 3 for `dim_product` and not Type 2 or Type 1
  - What role-playing dimensions are and how you used them